In [ ]:
# @title ⓪ Environment Setup (run once: env + models)

# @markdown ### 🔑 Authentication (Optional)
HF_TOKEN = "" # @param {type:"string"}
WANDB_API_KEY = "" # @param {type:"string"}

# @markdown ### Base model & compat
DIFFUSION_MODEL_NAME = "anima-base-v1.0.safetensors" # @param ["anima-base-v1.0.safetensors", "anima-aesthetic-v1.1.safetensors", "anima-aesthetic-v1.0b.safetensors", "anima-aesthetic-v1.0.safetensors", "anima-turbo-v1.0.safetensors", "anima-preview3-base.safetensors", "anima-preview2.safetensors", "anima-preview.safetensors"]
# @markdown Check `IS_T4_GPU` for Colab free-tier T4: disables FlashAttention, runs in compat mode.
IS_T4_GPU = True # @param {type:"boolean"}

# @markdown ### CJK Vocab Pack (`vocab_pack` in `configs/base.toml`)
# @markdown Upstream ships this ON (`models/vocab_packs/anima_cjk_vocab_pack_preview`). The loader auto-fetches it from `sorryhyun/anima-vocab-pack-cjk`, which 404s without repo access and then hard-aborts `preprocess-te` (`library/anima/vocab_pack.py:142`). Keep OFF unless the pack really downloads (HF_TOKEN with access, or files already in `models/vocab_packs/`). With it off, CJK captions encode with the stock tokenizer (EN is bit-exact either way).
# @markdown It lives HERE, not in Cell ①, because Cell ⓪ runs `git checkout -- configs/base.toml` before pulling, which resets the key to ON. `set_vocab_pack()` below re-pins it after every pull and before training.
USE_VOCAB_PACK = False # @param {type:"boolean"}

USE_GOOGLE_DRIVE = True

import os
import re
import sys
import glob
import shutil
import subprocess
_FP32_BLOCK_PATCHES = [
    (
        "1-RMSNorm.forward",
        """    def forward(self, x: torch.Tensor) -> torch.Tensor:
        output = self._norm(x.float())
        return (output * self.weight).to(x.dtype)
""",
        """    def forward(self, x: torch.Tensor) -> torch.Tensor:
        device_type = x.device.type if isinstance(x.device.type, str) and x.device.type != 'mps' else 'cpu'
        with torch.autocast(device_type=device_type, dtype=torch.float32):
            output = self._norm(x.float()).type_as(x)
            return output * self.weight
""",
    ),
    (
        "2-FinalLayer.forward",
        """    def forward(
        self,
        x_B_T_H_W_D: torch.Tensor,
        emb_B_T_D: torch.Tensor,
        adaln_lora_B_T_3D: Optional[torch.Tensor] = None,
    ):
        if self.use_adaln_lora:
            assert adaln_lora_B_T_3D is not None
""",
        """    def forward(
        self,
        x_B_T_H_W_D: torch.Tensor,
        emb_B_T_D: torch.Tensor,
        adaln_lora_B_T_3D: Optional[torch.Tensor] = None,
        use_fp32: bool = False,
    ):
        # fp32-block-restore: see restore_fp32_block_path() in the Colab notebook.
        if use_fp32:
            device_type = x_B_T_H_W_D.device.type if isinstance(x_B_T_H_W_D.device.type, str) and x_B_T_H_W_D.device.type != 'mps' else 'cpu'
            with torch.autocast(device_type=device_type, dtype=torch.float32):
                return self._forward_body(x_B_T_H_W_D, emb_B_T_D, adaln_lora_B_T_3D)
        return self._forward_body(x_B_T_H_W_D, emb_B_T_D, adaln_lora_B_T_3D)

    def _forward_body(
        self,
        x_B_T_H_W_D: torch.Tensor,
        emb_B_T_D: torch.Tensor,
        adaln_lora_B_T_3D: Optional[torch.Tensor] = None,
    ):
        if self.use_adaln_lora:
            assert adaln_lora_B_T_3D is not None
""",
    ),
    (
        "3-Block._forward",
        """    def _forward(
        self,
        x_B_T_H_W_D: torch.Tensor,
        emb_B_T_D: torch.Tensor,
        crossattn_emb: torch.Tensor,
        attn_params: attention_dispatch.AttentionParams,
        rope_cos_sin: Optional[tuple[torch.Tensor, torch.Tensor]] = None,
        adaln_lora_B_T_3D: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        if self.use_adaln_lora:
            fused_down = self.adaln_fused_down(emb_B_T_D)
""",
        """    def _forward(
        self,
        x_B_T_H_W_D: torch.Tensor,
        emb_B_T_D: torch.Tensor,
        crossattn_emb: torch.Tensor,
        attn_params: attention_dispatch.AttentionParams,
        rope_cos_sin: Optional[tuple[torch.Tensor, torch.Tensor]] = None,
        adaln_lora_B_T_3D: Optional[torch.Tensor] = None,
        use_fp32: bool = False,
    ) -> torch.Tensor:
        # fp32-block-restore: see restore_fp32_block_path() in the Colab notebook.
        if use_fp32:
            device_type = x_B_T_H_W_D.device.type if isinstance(x_B_T_H_W_D.device.type, str) and x_B_T_H_W_D.device.type != 'mps' else 'cpu'
            with torch.autocast(device_type=device_type, dtype=torch.float32):
                return self._forward_body(
                    x_B_T_H_W_D.float(), emb_B_T_D, crossattn_emb, attn_params,
                    rope_cos_sin, adaln_lora_B_T_3D,
                )
        return self._forward_body(
            x_B_T_H_W_D, emb_B_T_D, crossattn_emb, attn_params,
            rope_cos_sin, adaln_lora_B_T_3D,
        )

    def _forward_body(
        self,
        x_B_T_H_W_D: torch.Tensor,
        emb_B_T_D: torch.Tensor,
        crossattn_emb: torch.Tensor,
        attn_params: attention_dispatch.AttentionParams,
        rope_cos_sin: Optional[tuple[torch.Tensor, torch.Tensor]] = None,
        adaln_lora_B_T_3D: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        if self.use_adaln_lora:
            fused_down = self.adaln_fused_down(emb_B_T_D)
""",
    ),
    (
        "4-Block.forward",
        """    def forward(
        self,
        x_B_T_H_W_D: torch.Tensor,
        emb_B_T_D: torch.Tensor,
        crossattn_emb: torch.Tensor,
        attn_params: attention_dispatch.AttentionParams,
        rope_cos_sin: Optional[tuple[torch.Tensor, torch.Tensor]] = None,
        adaln_lora_B_T_3D: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        if torch.is_grad_enabled() and self.training and self.gradient_checkpointing:
            if self.unsloth_offload_checkpointing:
                return unsloth_checkpoint(
                    self._forward,
                    x_B_T_H_W_D,
                    emb_B_T_D,
                    crossattn_emb,
                    attn_params,
                    rope_cos_sin,
                    adaln_lora_B_T_3D,
                )
            else:
                return torch_checkpoint(
                    self._forward,
                    x_B_T_H_W_D,
                    emb_B_T_D,
                    crossattn_emb,
                    attn_params,
                    rope_cos_sin,
                    adaln_lora_B_T_3D,
                    use_reentrant=False,
                )
        else:
            return self._forward(
                x_B_T_H_W_D,
                emb_B_T_D,
                crossattn_emb,
                attn_params,
                rope_cos_sin,
                adaln_lora_B_T_3D,
            )
""",
        """    def forward(
        self,
        x_B_T_H_W_D: torch.Tensor,
        emb_B_T_D: torch.Tensor,
        crossattn_emb: torch.Tensor,
        attn_params: attention_dispatch.AttentionParams,
        rope_cos_sin: Optional[tuple[torch.Tensor, torch.Tensor]] = None,
        adaln_lora_B_T_3D: Optional[torch.Tensor] = None,
        use_fp32: bool = False,
    ) -> torch.Tensor:
        if torch.is_grad_enabled() and self.training and self.gradient_checkpointing:
            if self.unsloth_offload_checkpointing:
                # use_fp32 is passed POSITIONALLY on purpose: unsloth_checkpoint is
                # (function, *args) and accepts no **kwargs.
                return unsloth_checkpoint(
                    self._forward,
                    x_B_T_H_W_D,
                    emb_B_T_D,
                    crossattn_emb,
                    attn_params,
                    rope_cos_sin,
                    adaln_lora_B_T_3D,
                    use_fp32,
                )
            else:
                return torch_checkpoint(
                    self._forward,
                    x_B_T_H_W_D,
                    emb_B_T_D,
                    crossattn_emb,
                    attn_params,
                    rope_cos_sin,
                    adaln_lora_B_T_3D,
                    use_fp32,
                    use_reentrant=False,
                )
        else:
            return self._forward(
                x_B_T_H_W_D,
                emb_B_T_D,
                crossattn_emb,
                attn_params,
                rope_cos_sin,
                adaln_lora_B_T_3D,
                use_fp32,
            )
""",
    ),
    (
        "5a-marked_forward signature",
        """    def marked_forward(
        x_B_T_H_W_D,
        emb_B_T_D,
        crossattn_emb,
        attn_params,
        rope_cos_sin=None,
        adaln_lora_B_T_3D=None,
    ):
""",
        """    def marked_forward(
        x_B_T_H_W_D,
        emb_B_T_D,
        crossattn_emb,
        attn_params,
        rope_cos_sin=None,
        adaln_lora_B_T_3D=None,
        use_fp32=False,
    ):
""",
    ),
    (
        "5b-marked_forward forward-through",
        """        return compiled_inner(
            x_B_T_H_W_D,
            emb_B_T_D,
            crossattn_emb,
            attn_params,
            rope_cos_sin,
            adaln_lora_B_T_3D,
        )
""",
        """        return compiled_inner(
            x_B_T_H_W_D,
            emb_B_T_D,
            crossattn_emb,
            attn_params,
            rope_cos_sin,
            adaln_lora_B_T_3D,
            use_fp32,
        )
""",
    ),
    (
        "6a-_run_blocks signature",
        """        stop_after_block: Optional[int] = None,
        **block_kwargs,
    ) -> torch.Tensor:
""",
        """        stop_after_block: Optional[int] = None,
        use_fp32: bool = False,
        **block_kwargs,
    ) -> torch.Tensor:
""",
    ),
    (
        "6b-_run_blocks block() call",
        """            x = block(
                x,
                t_emb_block,
                crossattn_emb,
                attn_params,
                **block_kwargs,
            )
""",
        """            x = block(
                x,
                t_emb_block,
                crossattn_emb,
                attn_params,
                use_fp32=use_fp32,
                **block_kwargs,
            )
""",
    ),
    (
        "7a-Anima.forward use_fp32 origin",
        """        # Split point: pre/post-block regions stay eager, block loop is compiled.
        x_B_T_H_W_D = self._run_blocks(
            x_B_T_H_W_D,
            t_embedding_B_T_D,
            crossattn_emb,
            attn_params,
            capture_blocks=return_block_features,
            feature_sink=feature_sink,
            stop_after_block=stop_after_block,
            **block_kwargs,
        )
""",
        """        # fp32-block-restore: see restore_fp32_block_path() in the Colab notebook.
        # On fp16 (i.e. a T4, which has no bf16) this runs the whole block stack
        # under autocast(fp32); without it the late blocks overflow fp16's 65504
        # ceiling, the forward goes inf, and accelerate's GradScaler skips EVERY
        # optimizer step -- silently producing a bit-identical zero-weight LoRA.
        use_fp32 = x_B_T_H_W_D.dtype == torch.float16

        # Split point: pre/post-block regions stay eager, block loop is compiled.
        x_B_T_H_W_D = self._run_blocks(
            x_B_T_H_W_D,
            t_embedding_B_T_D,
            crossattn_emb,
            attn_params,
            capture_blocks=return_block_features,
            feature_sink=feature_sink,
            stop_after_block=stop_after_block,
            use_fp32=use_fp32,
            **block_kwargs,
        )
""",
    ),
    (
        "7b-Anima.forward final_layer call",
        """        x_B_T_H_W_O = self.final_layer(
            x_B_T_H_W_D, t_emb_final, adaln_lora_B_T_3D=adaln_lora_B_T_3D
        )
""",
        """        x_B_T_H_W_O = self.final_layer(
            x_B_T_H_W_D, t_emb_final, adaln_lora_B_T_3D=adaln_lora_B_T_3D, use_fp32=use_fp32
        )
""",
    ),
]

# ---------------------------------------------------------------------------
# Shared helpers (defined in Cell ⓪, consumed by Cell ⓪ / ① / ②)
# ---------------------------------------------------------------------------
LOG_DIR = "/content/anima_lora/output/notebook_logs"

VOCAB_PACK_PREFIX = "models/vocab_packs/anima_cjk_vocab_pack_preview"


def set_vocab_pack(enabled):
    """Pin `vocab_pack` in configs/base.toml. Called from EVERY cell that can
    precede training.

    Why this must be re-asserted rather than set once in Cell ①: Cell ⓪ runs
    `git checkout -- configs/base.toml` before pulling (to keep the tree clean),
    which silently restores the fork's shipped
    `vocab_pack = "models/vocab_packs/anima_cjk_vocab_pack_preview"` (ON). Train
    and `preprocess-te` would then disagree, and the only guard for that
    (`check_cache_stamp`) sits BEHIND the `skip_cache_check = true` early return
    in library/anima/strategy.py:270-271 -- so it never warns.
    """
    base_toml = "configs/base.toml"
    if not os.path.isfile(base_toml):
        print(f"⚠️ [config] {base_toml} not found; skipping vocab_pack pin")
        return
    import re as _re_vp

    want = VOCAB_PACK_PREFIX if enabled else '""'
    txt = open(base_toml, encoding="utf-8").read()
    new, n = _re_vp.subn(r"(?m)^vocab_pack\s*=.*$", f"vocab_pack = {want}", txt)
    if n == 0:
        new = txt.rstrip() + f"\nvocab_pack = {want}\n"
    if new != txt:
        with open(base_toml, "w", encoding="utf-8") as f:
            f.write(new)
    print(f"⚙️ [config] vocab_pack = {want}")
    if enabled:
        missing = [s for s in (".safetensors", ".json")
                   if not os.path.exists(VOCAB_PACK_PREFIX + s)]
        if missing:
            raise SystemExit(
                f"💥 USE_VOCAB_PACK=True but {VOCAB_PACK_PREFIX} is missing "
                f"{', '.join(missing)}.\n"
                "   Fetch it first: `uv run --no-sync make download-vocab-pack` "
                "(needs HF access to sorryhyun/anima-vocab-pack-cjk), or set\n"
                "   USE_VOCAB_PACK=False to encode with the stock tokenizer."
            )


FP32_RESTORE_MARKER = "fp32-block-restore"


def restore_fp32_block_path():
    """Re-introduce the `use_fp32` block-stack escape hatch into
    library/anima/models.py.

    Spr-Aachen/anima_lora @ HEAD removed it (their docs/optimizations/
    for_compile.md records "FinalLayer.forward: removed use_fp32 parameter and
    autocast wrapping entirely"; RMSNorm and Block lost it too). The
    Muffalo52 implementation computed

        use_fp32 = x_B_T_H_W_D.dtype == torch.float16

    in Anima.forward and threaded it down through _run_blocks -> Block.forward
    -> Block._forward -> FinalLayer, running the whole 28-block DiT under
    torch.autocast(dtype=torch.float32) on top of an explicit fp16 upcast.

    Why that matters on a T4: T4 has no bf16, so `--mixed_precision fp16` is the
    only option. Without the fp32 block stack the late-block residual stream
    exceeds fp16's 65504 ceiling (the fork's own docs/findings/selfflow.md:123
    records this), the forward goes inf, and accelerate's GradScaler silently
    SKIPS every optimizer step -- so a full run completes and emits a
    bit-identical ZERO-weight LoRA. Measured here: 364 lora_up tensors,
    sum|lora_up| == 0.0, identical across all 8 epochs.

    On a bf16-capable GPU `use_fp32` is always False, so this is a no-op there and
    can be dropped once you are off T4.

    Idempotent (guarded by FP32_RESTORE_MARKER) and MUST be re-run after every
    `git pull`, which overwrites the file. Any anchor that no longer matches
    raises instead of being skipped -- a silently-skipped anchor is exactly the
    failure mode this patch exists to fix.
    """
    target = "library/anima/models.py"
    if not os.path.isfile(target):
        raise SystemExit(f"💥 {target} not found (cwd={os.getcwd()}). Run Cell ⓪ first.")
    text = open(target, encoding="utf-8").read()
    if FP32_RESTORE_MARKER in text:
        print(f"  ⏭️ [Skip] {target} already carries the {FP32_RESTORE_MARKER} patch")
        _verify_fp32_block_path()
        return

    for name, old, new in _FP32_BLOCK_PATCHES:
        n = text.count(old)
        if n != 1:
            idx = text.find(old[:60])
            ctx = (text[max(0, idx - 200): idx + 500] if idx >= 0
                   else "<anchor head not found anywhere in the file>")
            raise SystemExit(
                f"💥 fp32-block-restore: anchor `{name}` matched {n} time(s), expected 1.\n"
                f"   The fork's models.py has drifted; this patch must be updated.\n"
                f"   ---- context around {target} ----\n{ctx}\n"
                f"   ---------------------------------------------"
            )
        text = text.replace(old, new, 1)
        print(f"  ✅ [OK] {name}")

    with open(target, "w", encoding="utf-8") as f:
        f.write(text)
    print(f"  ✅ [OK] wrote {target}")
    _verify_fp32_block_path()


_VERIFY_SNIPPET = """import inspect
import library.anima.models as M
# (class, method, needs_use_fp32?)
# RMSNorm.forward and the *_forward_body helpers are deliberately flagless: the
# former always uses the fp32 autocast context (as upstream did), the latter are
# the unconditional inner bodies the dispatchers delegate to.
targets = [(M.RMSNorm, "forward", False),
           (M.FinalLayer, "forward", True),
           (M.FinalLayer, "_forward_body", False),
           (M.Block, "forward", True),
           (M.Block, "_forward", True),
           (M.Block, "_forward_body", False),
           (M.Anima, "_run_blocks", True)]
bad = []
for cls, meth, wants in targets:
    fn = getattr(cls, meth, None)
    if fn is None:
        bad.append(f"{cls.__name__}.{meth}: MISSING")
    elif wants and "use_fp32" not in inspect.signature(fn).parameters:
        bad.append(f"{cls.__name__}.{meth}: no use_fp32 parameter")
print("VERIFY_BAD=" + ("none" if not bad else " ; ".join(bad)))
"""


def _verify_fp32_block_path():
    """Import the patched module with the SAME interpreter training will use
    (.venv/bin/python -- the notebook kernel is Colab's system python, which
    cannot import the repo's deps) and assert `use_fp32` reached every signature
    the block stack threads it through. Catches a partial patch now rather than
    as a TypeError 4000 steps into training."""
    py = os.path.join(".venv", "bin", "python")
    if not os.path.isfile(py):
        print("  ⚠️ [skip] .venv not present yet; signature check deferred to Cell ②")
        return
    os.makedirs(LOG_DIR, exist_ok=True)
    probe = os.path.join(LOG_DIR, "_fp32_check.py")
    with open(probe, "w", encoding="utf-8") as f:
        f.write(_VERIFY_SNIPPET)
    proc = subprocess.run([py, probe], capture_output=True, text=True,
                          cwd=os.getcwd(), timeout=300)
    line = next((l for l in (proc.stdout or "").splitlines()
                 if l.startswith("VERIFY_BAD=")), "")
    if proc.returncode != 0 or not line:
        print("  ⚠️ [warn] signature check could not run:\n"
              + ((proc.stderr or "")[-800:]))
        return
    if line != "VERIFY_BAD=none":
        raise SystemExit(
            "💥 fp32-block-restore verification FAILED:\n   "
            + line[len("VERIFY_BAD="):]
            + "\n   The patch did not apply cleanly. Do NOT start training."
        )
    print("  ✅ [OK] use_fp32 present on RMSNorm/FinalLayer/Block/Anima signatures")


def run_step(label, cmd, hint=""):
    """Run a step, streaming to the console AND to output/notebook_logs/<step>.log,
    raising on non-zero exit.

    Two reasons this exists (both learned the hard way):
      * The IPython `!` magic swallows the exit code, so a crashed step silently
        falls through to the next one. That is how a zero-weight LoRA got saved
        after a "successful-looking" run.
      * Scrolling up in Colab loses the real traceback.
    """
    os.makedirs(LOG_DIR, exist_ok=True)
    log_path = os.path.join(LOG_DIR, f"{label}.log".replace(" ", "_"))
    print(f"   (log: {log_path})")
    proc = subprocess.Popen(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
        text=True, encoding="utf-8", errors="replace", bufsize=1,
    )
    tail = []
    with open(log_path, "w", encoding="utf-8") as log:
        for line in proc.stdout:
            sys.stdout.write(line)
            log.write(line)
            tail.append(line.rstrip())
            if len(tail) > 60:
                tail.pop(0)
    proc.wait()
    if proc.returncode != 0:
        raise SystemExit(
            f"💥 {label} failed (exit {proc.returncode}).\n"
            f"   command: {' '.join(cmd)}\n"
            f"   {hint}\n"
            f"   ---- last {len(tail)} line(s) of {log_path} ----\n"
            + "\n".join(tail)
            + "\n   --------------------------------------------"
        )


def assert_lora_actually_trained(output_dir):
    """Fail loudly if the saved LoRA is still at its zero-init.

    `lora_up` is zero-initialised, so a run whose every optimizer step was
    skipped by the GradScaler still writes a perfectly plausible safetensors
    file full of exact zeros -- which renders as "the LoRA does nothing" at
    inference, with no error anywhere. Turn that silent failure into a hard stop.
    """
    from safetensors.torch import load_file
    import torch

    files = sorted(glob.glob(os.path.join(output_dir, "*.safetensors")))
    if not files:
        raise SystemExit(f"💥 No .safetensors under {output_dir} -- training saved nothing.")
    latest = files[-1]
    sd = load_file(latest)
    ups = [k for k in sd if k.endswith("lora_up.weight")]
    if not ups:
        raise SystemExit(f"💥 {latest} contains no lora_up.weight tensors -- not a LoRA?")
    mag = sum(float(sd[k].float().abs().sum()) for k in ups)
    print(f"🔍 [LoRA check] {os.path.basename(latest)}: {len(ups)} lora_up tensors, "
          f"sum|lora_up| = {mag:.6g}")
    if mag == 0.0:
        raise SystemExit(
            "💥 The saved LoRA is exactly zero -- NOT ONE optimizer step landed.\n"
            "   This is the GradScaler-skip signature: the forward went inf/nan and\n"
            "   accelerate silently dropped every step. Check train.log for a nan loss.\n"
            "   On a T4 it means the fp32 block stack is missing -- verify that\n"
            "   restore_fp32_block_path() ran in Cell ② before training."
        )
    if len(files) > 1:
        prev = load_file(files[0])
        same = all(k in prev and torch.equal(prev[k].float(), sd[k].float()) for k in ups)
        if same:
            raise SystemExit(
                f"💥 {os.path.basename(files[0])} and {os.path.basename(latest)} are "
                "byte-identical -- training made no progress at all."
            )
    return mag

# 1. Mount Drive
if USE_GOOGLE_DRIVE:
    if not os.path.exists('/content/drive'):
        from google.colab import drive
        drive.mount('/content/drive')

# 2. Set env vars
os.environ['PATH'] = f"/root/.local/bin:{os.environ['PATH']}"
os.environ['PYTHONWARNINGS'] = "ignore"
if HF_TOKEN.strip():
    os.environ['HF_TOKEN'] = HF_TOKEN

# 3 & 4. Set up env and install deps (idempotent: skipped if present)
print("\n=== [1/3] Environment Setup and Dependencies ===")
REPO_URL = "https://github.com/Spr-Aachen/anima_lora.git"
TARGET_COMMIT = "HEAD" # Change to a specific commit hash if needed
if shutil.which("aria2c") is None:
    !apt install -y pigz aria2 -qq
if shutil.which("uv") is None:
    !curl -LsSf https://astral.sh/uv/install.sh | sh
    !uv python install 3.13

if not os.path.exists('/content/anima_lora'):
    print(f"Cloning: {REPO_URL}")
    subprocess.run(["git", "clone", REPO_URL, "/content/anima_lora"], check=True)
if not os.path.exists('/content/anima_lora'):
    raise SystemExit(f"💥 git clone failed: /content/anima_lora not found (REPO_URL={REPO_URL}). Check network/repo access.")
os.chdir('/content/anima_lora')
print(f"📁 CWD: {os.getcwd()}")
# Show current checkout so a stale clone is visible (Cell ⓪ never re-clones).
try:
    _log = subprocess.run(["git", "log", "--oneline", "-3"], capture_output=True, text=True)
    print("📌 [Repo] current checkout:")
    print((_log.stdout or "").strip() or "(no git log available)")
except Exception as e:
    print(f"⚠️ [Repo] git log failed: {e}")
# Cells ①/② rewrite configs/base.toml; drop that edit first so the ff-only
# pull below cannot be blocked by a dirty working tree. NOTE this also resets
# `vocab_pack` to the fork's shipped ON value -- set_vocab_pack() below re-pins it.
if os.path.exists('configs/base.toml'):
    subprocess.run(["git", "checkout", "--", "configs/base.toml"], capture_output=True)

# A bare REPO_URL change never re-clones (guard above), so pull when .git exists.
if os.path.exists('/content/anima_lora/.git'):
    try:
        print("🔄 [Repo] pulling latest (git pull --ff-only)...")
        _pull = subprocess.run(["git", "pull", "--ff-only"], capture_output=True, text=True)
        print((_pull.stdout or "").strip() or "(already up to date)")
        if _pull.returncode != 0:
            print(f"⚠️ [Repo] git pull failed (exit {_pull.returncode}): {(_pull.stderr or '').strip()[:500]}")
            print("   If you changed REPO_URL, delete /content/anima_lora first and re-run Cell ⓪.")
    except Exception as e:
        print(f"⚠️ [Repo] git pull error: {e}")

# `git pull` overwrites library/anima/models.py, so the fp32 block-stack patch
# must be re-applied on EVERY Cell ⓪ run, not just the first one.
print("\n=== [1/4] Repo-local patches ===")
restore_fp32_block_path()
set_vocab_pack(USE_VOCAB_PACK)

if TARGET_COMMIT != "HEAD":
    print(f"Checking out: {TARGET_COMMIT}")
    subprocess.run(["git", "checkout", TARGET_COMMIT], check=True)

if os.path.exists('.venv/bin/python'):
    print("Valid virtual environment (.venv) found. Skipping installation.")
else:
    print("Installing environment...")
    !uv venv --python 3.13 --clear
    # `uv sync` validates EVERY source in [tool.uv.sources], including the
    # group-conditioned `anime-tools-dev` editable path `../anime_tools`
    # that only exists in a sibling dev checkout. On a single-repo clone
    # (Colab) that aborts with
    #   error: Failed to generate package metadata for
    #     `anime-tools==0.7.5 @ editable+../anime_tools`
    # even though `anime-tools-dev` is not an enabled group (the default
    # group `anime-tools-git` is what actually gets installed).
    # `--frozen` skips re-locking and installs straight from the committed
    # uv.lock, which pins anime-tools to the upstream git tag.
    _missing_sibling_tools = (
        os.path.isfile("pyproject.toml")
        and 'path = "../anime_tools"' in open("pyproject.toml", encoding="utf-8").read()
        and not os.path.isdir(os.path.join(os.path.dirname(os.path.abspath(".")), "anime_tools"))
    )
    _sync_cmd = ["uv", "sync"] + (["--frozen"] if _missing_sibling_tools else [])
    if _missing_sibling_tools:
        print("   (no sibling ../anime_tools checkout -> `uv sync --frozen`, using the committed uv.lock)")
    _sync = subprocess.run(_sync_cmd)
    if _sync.returncode != 0:
        raise SystemExit(
            f"💥 `{' '.join(_sync_cmd)}` failed (exit {_sync.returncode}). The real error is in the output above.\n"
            "If it mentions `../anime_tools`, the clone is missing its sibling dev checkout; delete /content/anima_lora and re-run Cell ⓪.\n"
            "If it is a network/download failure, re-run Cell ⓪ (nothing else was touched)."
        )

# 5. Download model weights (per-file existence check)
print("\n=== [1/3] Downloading Pre-trained Models ===")
downloads = [
    {"url": "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/vae/qwen_image_vae.safetensors", "dest": "models/vae/qwen_image_vae.safetensors"},
    {"url": "https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/text_encoders/qwen_3_06b_base.safetensors", "dest": "models/text_encoders/qwen_3_06b_base.safetensors"},
    {"url": f"https://huggingface.co/circlestone-labs/Anima/resolve/main/split_files/diffusion_models/{DIFFUSION_MODEL_NAME}", "dest": f"models/diffusion_models/{DIFFUSION_MODEL_NAME}"}
]

max_retries = 3
timeout_seconds = 120

for item in downloads:
    dest_path = item["dest"]
    url = item["url"]
    os.makedirs(os.path.dirname(dest_path), exist_ok=True)
    if not os.path.exists(dest_path):
        print(f"⬇️ Downloading: {os.path.basename(dest_path)}")
        aria2_cmd = ["aria2c", "--console-log-level=warn", "-c", "-s", "16", "-x", "16", "-k", "10M", "-d", os.path.dirname(dest_path), "-o", os.path.basename(dest_path)]
        if HF_TOKEN.strip():
            aria2_cmd.extend(["--header", f"Authorization: Bearer {HF_TOKEN.strip()}"])
        aria2_cmd.append(url)

        success = False
        for attempt in range(1, max_retries + 1):
            try:
                subprocess.run(aria2_cmd, check=True, timeout=timeout_seconds)
                success = True
                break
            except subprocess.TimeoutExpired:
                print(f"⚠️ Download timed out (2 min). Retrying... ({attempt}/{max_retries})")
            except subprocess.CalledProcessError as e:
                print(f"⚠️ Error ({e}). Retrying... ({attempt}/{max_retries})")

        if not success:
            print(f"\n💥 Error: aria2c download failed for {os.path.basename(dest_path)}.")
            raise SystemExit("Download failed, aborting.")
    else:
        print(f"✅ Already exists: {os.path.basename(dest_path)}")

print("✅ Model download complete.")

print("\n✅ Env ready. Next: run Cell ① (Dataset & Preprocess).")

In [ ]:
# @title ① Dataset & Preprocess (re-run when dataset changes)

# @markdown Enter the folder or zip filename under `MyDrive/lora_training/datasets/`.
# @markdown Optional: set `IMPORT_ZIP_PATH` to a zip anywhere on Drive to import it into `datasets/{PROJECT_NAME}` on run. Leave empty to use the folder/zip already in place.
# @markdown Subfolders are supported (`recursive=true`): `good/`, `normal/`, ... are kept through resize/cache. Prefix a folder with `N_` (e.g. `3_good/`, `1_normal/`, `0_skip/`) to repeat/drop it per epoch (`repeat_by_folder_name=true` in `configs/base.toml`). Priority: `N_` folder overrides `NUM_REPEATS` below (override, not multiply); `0_` always drops.
USE_GOOGLE_DRIVE = True # @param {type:"boolean"}
PROJECT_NAME = "" # @param {type:"string"}
FORCE_REPREPROCESS = False # @param {type:"boolean"}
IMPORT_ZIP_PATH = "" # @param {type:"string"}
# @markdown Global repeats per image per epoch (SD-style `num_repeats`). Effective = `N_` folder if present, else `NUM_REPEATS`.
NUM_REPEATS = 9 # @param {type:"integer"}
# @markdown Training resolution pool, space-separated. (Allowed: 512 768 896 1024 1280 1536)
TARGET_RES = "512 768 896 1024" # @param {type:"string"}
MIXED_PRECISION = "fp16"

# @markdown Captions are required: every image needs a hand-written `.txt` file (same basename).
REQUIRE_CAPTIONS = True # @param {type:"boolean"}

# @markdown ### SAM3 Masking (Masked Loss)
# @markdown Enable `USE_MASKED_LOSS` to exclude background/noise from loss per prompts.
USE_MASKED_LOSS = False # @param {type:"boolean"}
MASK_THRESHOLD = 0.7
# @markdown Elements to ignore (e.g. `text, speech bubble, watermark`), comma-separated.
MASK_IGNORE_PROMPTS = "" # @param {type:"string"}
# @markdown Focus targets (e.g. `1girl, character`). Empty = only remove ignored elements.
MASK_FOCUS_PROMPTS = "" # @param {type:"string"}
USE_SIGMA_LOWRES = True
USE_REPA = False
USE_CMMD = False
# @markdown ### CJK Vocab Pack
# @markdown `USE_VOCAB_PACK` is set in Cell ⓪ (single source of truth: Cell ⓪ resets `configs/base.toml` on every pull). It is re-pinned here through `set_vocab_pack()` so `preprocess-te` and `train` can never disagree.

import os
import shutil
import subprocess
import glob
import gc
import sys
import torch

# Carry base-model choice from Cell ⓪ (fallback for standalone runs)
if "DIFFUSION_MODEL_NAME" not in globals():
    DIFFUSION_MODEL_NAME = "anima-base-v1.0.safetensors"
# Carry T4 flag from Cell ⓪ (fallback for standalone runs)
if "IS_T4_GPU" not in globals():
    IS_T4_GPU = True
if "USE_VOCAB_PACK" not in globals():
    USE_VOCAB_PACK = False
assert os.path.exists("/content/anima_lora"), "Run Cell ⓪ (Environment Setup) first."
os.chdir("/content/anima_lora")

# --- CJK vocab pack -------------------------------------------------------
# `preprocess-te` and `train` both read `vocab_pack` from configs/base.toml, and a
# disagreement silently poisons the TE caches: the only guard for that
# (`check_cache_stamp`) sits behind the `skip_cache_check = true` early return in
# library/anima/strategy.py:270-271, so it never warns. set_vocab_pack() is defined
# in Cell ⓪ and re-called here, in Cell ②, and after every `git pull`.
set_vocab_pack(USE_VOCAB_PACK)

# Remount guard: runtime may have restarted since Cell ⓪ (same pattern as SD trainer)
if USE_GOOGLE_DRIVE and not os.path.exists("/content/drive"):
    from google.colab import drive
    print("\U0001f4c2 Reconnecting Google Drive...")
    drive.mount("/content/drive")

# Path setup
PROJECT_NAME = (PROJECT_NAME or "").strip()
assert PROJECT_NAME, 'Set PROJECT_NAME (folder or zip name under MyDrive/lora_training/datasets/).'
assert not any(c in PROJECT_NAME for c in ' .()\"\'/\\'), f'Invalid PROJECT_NAME "{PROJECT_NAME}": spaces and .()\"\'/\\ are not allowed.'
DRIVE_BASE = "/content/drive/MyDrive/lora_training"
DATASET_FOLDER = os.path.join(DRIVE_BASE, "datasets", PROJECT_NAME)
DATASET_ZIP = DATASET_FOLDER + ".zip"
OUTPUT_DIR = os.path.join(DRIVE_BASE, "output", PROJECT_NAME)
LOCAL_DATASET = "/content/anima_lora/image_dataset"
MARKER_FILE = os.path.join("/content/anima_lora", f".cache_done_{PROJECT_NAME}")


if FORCE_REPREPROCESS:
    existing_markers = glob.glob("/content/anima_lora/.cache_done_*")
    for marker in existing_markers:
        try:
            os.remove(marker)
            print(f"🧹 [FORCE_REPREPROCESS=True] marker removed: {os.path.basename(marker)}")
        except Exception as e:
            pass
    if os.path.exists(LOCAL_DATASET):
        shutil.rmtree(LOCAL_DATASET)
        print(f"🧹 [FORCE_REPREPROCESS=True] cleared local dataset folder ({LOCAL_DATASET}).")
    if os.path.exists("post_image_dataset"):
        shutil.rmtree("post_image_dataset")
        print(f"🧹 [FORCE_REPREPROCESS=True] cleared preprocess cache folder (post_image_dataset).")

# Auto-skip when marker file exists
SKIP_PREPROCESSING = os.path.exists(MARKER_FILE) and not FORCE_REPREPROCESS

os.makedirs(OUTPUT_DIR, exist_ok=True)

# 6. Dataset setup
print("\n=== [2/3] Dataset Preprocessing ===")
IMAGE_EXTS = ('.png', '.jpg', '.jpeg', '.webp', '.bmp')
CAPTION_EXTS = ('.txt', '.npz')

def _has_images_recursive(target_dir):
    if not os.path.exists(target_dir):
        return False
    return any(
        f.lower().endswith(IMAGE_EXTS)
        for _, _, files in os.walk(target_dir)
        for f in files
    )

def _normalize_extracted_tree(root):
    # A zip often wraps everything in one top-level folder:
    #   LOCAL_DATASET/<wrapper>/*.png  ->  LOCAL_DATASET/*.png
    # Only lift when root itself has no images and exactly one subdir;
    # multi-folder zips (good/, normal/, ...) are left untouched.
    if not os.path.isdir(root):
        return
    top_images = [f for f in os.listdir(root)
                  if os.path.isfile(os.path.join(root, f)) and f.lower().endswith(IMAGE_EXTS)]
    subdirs = [d for d in os.listdir(root) if os.path.isdir(os.path.join(root, d))]
    if top_images or len(subdirs) != 1:
        return
    wrapper = os.path.join(root, subdirs[0])
    if not _has_images_recursive(wrapper):
        return
    import tempfile
    tmp = root + "__unwrap_tmp"
    if os.path.exists(tmp):
        shutil.rmtree(tmp)
    os.rename(wrapper, tmp)
    shutil.rmtree(root)
    os.rename(tmp, root)
    print(f"📦 Unwrapped single top-level folder '{subdirs[0]}/' from archive.")

# 5b. Optional import: zip from anywhere on Drive -> datasets/{PROJECT_NAME}
_IMPORT_ZIP = (IMPORT_ZIP_PATH or "").strip()
if _IMPORT_ZIP:
    if not _IMPORT_ZIP.startswith(("/content/", "/")):
        _IMPORT_ZIP = os.path.join("/content/drive/MyDrive", _IMPORT_ZIP)
    if not os.path.isfile(_IMPORT_ZIP):
        raise SystemExit(
            f"\U0001f4a5 IMPORT_ZIP_PATH not found: {_IMPORT_ZIP}\n"
            "Check that the path is a full Drive mount path (/content/drive/MyDrive/...)."
        )
    _IMPORT_TMP = DATASET_FOLDER + "__import_tmp"
    if os.path.exists(DATASET_FOLDER):
        if not FORCE_REPREPROCESS:
            raise SystemExit(
                f"\U0001f4a5 Target already exists: {DATASET_FOLDER}\n"
                "Tick FORCE_REPREPROCESS=True to clear and re-import, or empty IMPORT_ZIP_PATH to use the existing data."
            )
        print(f"\U0001f9f9 [FORCE_REPREPROCESS=True] removing existing dataset: {DATASET_FOLDER}")
        shutil.rmtree(DATASET_FOLDER)
    if os.path.exists(_IMPORT_TMP):
        shutil.rmtree(_IMPORT_TMP)
    print(f"\U0001f4e6 Importing: {_IMPORT_ZIP}\n   -> {DATASET_FOLDER}")
    try:
        shutil.unpack_archive(_IMPORT_ZIP, _IMPORT_TMP)
    except shutil.ReadError as e:
        raise SystemExit(f"\U0001f4a5 Not a supported archive (.zip/.tar/.gztar): {_IMPORT_ZIP}\n{e}")
    _normalize_extracted_tree(_IMPORT_TMP)
    _n_imp = sum(1 for _, _, _fs in os.walk(_IMPORT_TMP) for _f in _fs if _f.lower().endswith(IMAGE_EXTS))
    if _n_imp == 0:
        shutil.rmtree(_IMPORT_TMP)
        raise SystemExit(f"\U0001f4a5 No training images (.png/.jpg/.jpeg/.webp/.bmp) in {_IMPORT_ZIP}. Check the archive contents.")
    os.rename(_IMPORT_TMP, DATASET_FOLDER)
    print(f"\u2705 Imported {_n_imp} images -> {DATASET_FOLDER}")
    # Invalidate local cache so the fresh data is copied + preprocessed below
    for _m in glob.glob("/content/anima_lora/.cache_done_*"):
        try:
            os.remove(_m)
        except Exception:
            pass
    if os.path.exists(LOCAL_DATASET):
        shutil.rmtree(LOCAL_DATASET)
    if os.path.exists("post_image_dataset"):
        shutil.rmtree("post_image_dataset")
    SKIP_PREPROCESSING = False
    print("\U0001f9f9 Local cache cleared for re-import.")

if USE_GOOGLE_DRIVE:
    if _has_images_recursive(LOCAL_DATASET):
        print(f"✅ Local dataset ({LOCAL_DATASET}) exists, skipping Drive copy.")
    else:
        if os.path.exists(LOCAL_DATASET):
            shutil.rmtree(LOCAL_DATASET)
        if os.path.exists(DATASET_FOLDER):
            print(f"Copying folder: {DATASET_FOLDER}")
            shutil.copytree(DATASET_FOLDER, LOCAL_DATASET)
        elif os.path.exists(DATASET_ZIP):
            print(f"Extracting archive: {DATASET_ZIP}")
            shutil.unpack_archive(DATASET_ZIP, LOCAL_DATASET)
            _normalize_extracted_tree(LOCAL_DATASET)
        else:
            raise SystemExit(
                f"💥 Dataset not found: neither folder nor zip exists.\n"
                f"   folder: {DATASET_FOLDER}\n"
                f"   zip:    {DATASET_ZIP}\n"
                f"Put your images (or a .zip of them) under MyDrive/lora_training/datasets/ and re-run. "
                f"If your zip lives elsewhere on Drive, set IMPORT_ZIP_PATH to its Drive path and re-run."
            )

def count_dataset_images(target_dir):
    # Raw (unweighted) image count, kept for reference.
    count = 0
    if os.path.exists(target_dir):
        for _, _, files in os.walk(target_dir):
            for f in files:
                if f.lower().endswith(IMAGE_EXTS):
                    count += 1
    return count

def _folder_repeat_of_rel(rel_dir):
    # Backend-compatible: deepest {n}_ component wins (mirrors folder_repeat_count).
    # Returns None when no N_ prefix -> caller falls back to NUM_REPEATS.
    import re as _re
    if not rel_dir or rel_dir == "." or rel_dir.startswith(".."):
        return None
    for part in reversed(_re.split(r"[\\/]", rel_dir)):
        m = _re.match(r"^(\d+)_.*$", part)
        if m:
            return int(m.group(1))
    return None

def count_effective_images(target_dir, global_repeats=1):
    # Weighted count: N_ folder overrides global_repeats (override, not multiply); 0_ drops.
    eff = 0
    if os.path.exists(target_dir):
        for root, _, files in os.walk(target_dir):
            rel = os.path.relpath(root, target_dir)
            rep = _folder_repeat_of_rel(rel)
            if rep is None:
                rep = global_repeats
            if rep == 0:
                continue
            for f in files:
                if f.lower().endswith(IMAGE_EXTS):
                    eff += rep
    return eff

def summarize_dataset_by_folder(target_dir, global_repeats=1):
    # Per top-level folder table (root images go under "."): images / captions / repeat / effective steps.
    # Effective per file = N_ folder (deepest wins) if present else global_repeats. Returns (stats, raw_total, effective_total).
    import re
    stats = {}
    if os.path.exists(target_dir):
        for root, _, files in os.walk(target_dir):
            rel = os.path.relpath(root, target_dir)
            top = "." if rel == "." else rel.split(os.sep)[0]
            s = stats.setdefault(top, {"img": 0, "cap": 0, "other": 0, "eff": 0})
            rep = _folder_repeat_of_rel(rel)
            if rep is None:
                rep = global_repeats
            for f in files:
                fl = f.lower()
                if fl.endswith(IMAGE_EXTS):
                    s["img"] += 1
                    if rep != 0:
                        s["eff"] += rep
                elif fl.endswith(CAPTION_EXTS):
                    s["cap"] += 1
                else:
                    s["other"] += 1
    if not stats:
        return {}, 0, 0
    def _repeat_of(name):
        m = re.match(r"^(\d+)_.*$", name)
        return int(m.group(1)) if m else global_repeats
    print(f"\n📁 [Dataset folders] (N_ prefix overrides NUM_REPEATS={global_repeats}, 0_ = dropped)")
    width = max(len(k) for k in stats)
    raw_total = 0
    eff_total = 0
    for k in sorted(stats):
        s = stats[k]
        rep = _repeat_of(k) if k != "." else global_repeats
        eff = s["eff"]
        raw_total += s["img"]
        eff_total += eff
        extra = f" x{rep} -> {eff} eff" if k != "." and rep != global_repeats else f" x{rep} (global) -> {eff} eff" if k == "." else f" -> {eff} eff"
        other = f" | {s['other']} other" if s["other"] else ""
        print(f"  📁 {k.ljust(width)} | {s['img']:>4} images | {s['cap']:>4} captions{other}{extra}")
    print(f"  📊 raw={raw_total} images | effective={eff_total} samples/epoch (global NUM_REPEATS={global_repeats})")
    return stats, raw_total, eff_total

def warn_unknown_files(target_dir):
    bad = []
    if os.path.exists(target_dir):
        for root, _, files in os.walk(target_dir):
            for f in files:
                fl = f.lower()
                if not fl.endswith(IMAGE_EXTS + CAPTION_EXTS):
                    bad.append(os.path.join(os.path.relpath(root, target_dir), f))
    if bad:
        print(f"\n⚠️ {len(bad)} non-dataset file(s) will be ignored by preprocessing (only {IMAGE_EXTS} + {CAPTION_EXTS} are used):")
        for b in sorted(bad)[:20]:
            print(f"   - {b}")
        if len(bad) > 20:
            print(f"   ... and {len(bad) - 20} more")

NUM_REPEATS = int(globals().get("NUM_REPEATS", 10))
if NUM_REPEATS < 0:
    NUM_REPEATS = 0
SCAN_DIR = LOCAL_DATASET if _has_images_recursive(LOCAL_DATASET) else DATASET_FOLDER
NUM_IMAGES_RAW = count_dataset_images(SCAN_DIR)
NUM_IMAGES_EFFECTIVE = count_effective_images(SCAN_DIR, NUM_REPEATS)
NUM_IMAGES = NUM_IMAGES_EFFECTIVE
print(f"\n\U0001f4ca [Dataset] raw={NUM_IMAGES_RAW} images | effective={NUM_IMAGES_EFFECTIVE} samples/epoch (NUM_REPEATS={NUM_REPEATS}) in project \"{PROJECT_NAME}\"")
FOLDER_STATS, _, _ = summarize_dataset_by_folder(SCAN_DIR, NUM_REPEATS)
warn_unknown_files(LOCAL_DATASET if os.path.exists(LOCAL_DATASET) else DATASET_FOLDER)
if NUM_IMAGES == 0:
    raise SystemExit(
        f"💥 No training images found for project \"{PROJECT_NAME}\" (checked {LOCAL_DATASET} and {DATASET_FOLDER}). "
        "Add .png/.jpg/.jpeg/.webp/.bmp files and re-run with FORCE_REPREPROCESS=True."
    )

def require_captions():
    img_cnt = count_dataset_images(LOCAL_DATASET)

    valid_exts = ('.png', '.jpg', '.jpeg', '.webp', '.bmp', '.PNG', '.JPG', '.JPEG', '.WEBP', '.BMP')
    txt_cnt = 0
    if os.path.exists(LOCAL_DATASET):
        for root, _, files in os.walk(LOCAL_DATASET):
            for f in files:
                if f.endswith(valid_exts):
                    base_name = os.path.splitext(f)[0]
                    if os.path.exists(os.path.join(root, base_name + ".txt")):
                        txt_cnt += 1

    print(f"\n🔍 [Tag check] images: {img_cnt} / `.txt` tags: {txt_cnt}")

    if not REQUIRE_CAPTIONS:
        print("⚠️ REQUIRE_CAPTIONS=False, skipping caption check (TE caches may be empty).")
        return
    if img_cnt > 0 and txt_cnt < img_cnt:
        missing = img_cnt - txt_cnt
        raise SystemExit(
            f"💥 {missing}/{img_cnt} images have no `.txt` caption. "
            "Add `.txt` files next to the images (same basename) and re-run with FORCE_REPREPROCESS=True."
        )
    print("✅ .txt tags found for all images.")

def _flash_attn_spec_in_venv():
    # Probe .venv/bin/python directly (NOT `uv run`, NOT the notebook kernel:
    # Colab's system python and .venv are different environments).
    # Returns e.g. 'None' when absent, or 'UNKNOWN(...)' when the probe itself fails.
    try:
        _p = subprocess.run(
            [".venv/bin/python", "-c", "import importlib.util; print(importlib.util.find_spec('flash_attn'))"],
            capture_output=True, text=True, timeout=120,
        )
        if _p.returncode != 0:
            return f"UNKNOWN(probe exit {_p.returncode}: {(_p.stderr or '').strip()[:300]})"
        return (_p.stdout or "").strip()
    except Exception as e:
        return f"UNKNOWN({e})"

def ensure_t4_compat():
    # T4 (Turing) has no FlashAttention support (Ampere+ only), but the TE
    # step's LLMAdapter uses flash whenever the package is importable
    # (networks/attention_dispatch + library/anima/models.py LLMAdapterAttention:
    # `can_use_flash = flash_attn_varlen_func is not None and dtype in (fp16, bf16)`).
    # Removing the package forces the SDPA fallback. Must run BEFORE any
    # preprocess step (Cell ② does the same for training-only runs).
    # NOTE: only `uv pip uninstall` here — never `uv remove` (that edits
    # pyproject.toml, so the next `uv sync` would behave unpredictably).
    if not globals().get("IS_T4_GPU", True):
        return
    _spec = _flash_attn_spec_in_venv()
    if _spec == "None":
        print("\n✅ [T4 compat] flash-attn not in .venv, using SDPA.")
        return
    print(f"\n🚀 [T4 compat] flash-attn found in .venv on T4 (Turing): {_spec}")
    print("   removing so preprocess falls back to SDPA...")
    _un = subprocess.run(
        ["uv", "pip", "uninstall", "--python", ".venv/bin/python", "-y", "flash-attn"],
        capture_output=True, text=True, timeout=300,
    )
    print(((_un.stdout or "") + "\n" + (_un.stderr or "")).strip()[-2000:])
    if _un.returncode != 0:
        raise SystemExit(f"💥 [T4 compat] failed to uninstall flash-attn (exit {_un.returncode}). Fix manually, then re-run Cell ①.")
    _spec2 = _flash_attn_spec_in_venv()
    if _spec2 != "None":
        raise SystemExit(f"💥 [T4 compat] flash-attn still importable after uninstall: {_spec2}. Run `uv pip uninstall --python .venv/bin/python -y flash-attn` manually, verify find_spec is None, then re-run Cell ①.")
    print("  ✅ flash-attn removed and verified (SDPA fallback active)")

T4_HINT = (
    "On a T4 (16GB) this is usually CUDA OOM in the VAE/text-encoder: lower "
    "TARGET_RES (e.g. '512 768 896') and/or set USE_SIGMA_LOWRES=False, then re-run "
    "Cell ① with FORCE_REPREPROCESS=True."
)
FLASH_HINT = (
    "If it is 'FlashAttention only supports Ampere GPUs or newer', the [T4 compat] "
    "step did not take effect: run `uv pip uninstall --python .venv/bin/python -y "
    "flash-attn`, verify find_spec is None, then re-run Cell ① (no FORCE_REPREPROCESS "
    "needed)."
)

def run_preprocess():
    if not SKIP_PREPROCESSING:
        ensure_t4_compat()
        print("\n📦 Downloading Danbooru Tags...")
        run_step("download-danbooru-tags", ["uv", "run", "--no-sync", "python", "tasks.py", "download-danbooru-tags"])

        require_captions()

        print("\n🚀 Running image bucketing & VAE latent caching...")
        run_step("preprocess-resize", ["uv", "run", "--no-sync", "make", "preprocess-resize", f"ARGS=--target_res {TARGET_RES}"])

        # SAM3 masking
        if USE_MASKED_LOSS:
            print("\n🎭 Preparing SAM3 Model & Generating Masks...")
            os.makedirs("models/sam3", exist_ok=True)
            files_to_download = {
                "sam3.pt": "https://huggingface.co/1038lab/sam3/resolve/main/sam3.pt",
                "config.json": "https://huggingface.co/AEmotionStudio/sam3/resolve/main/config.json"
            }
            for filename, url in files_to_download.items():
                dest = os.path.abspath(f"models/sam3/{filename}")
                if not os.path.exists(dest):
                    print(f"⬇️ Downloading {filename} via aria2c...")
                    aria2_cmd = ["aria2c", "--console-log-level=warn", "-c", "-s", "16", "-x", "16", "-d", "models/sam3", "-o", filename, url]
                    subprocess.run(aria2_cmd, check=True)

            ignore_list = [f'"{p.strip()}"' for p in MASK_IGNORE_PROMPTS.split(',') if p.strip()]
            focus_list = [f'"{p.strip()}"' for p in MASK_FOCUS_PROMPTS.split(',') if p.strip()]
            mask_yaml_content = f"prompts: [{ ', '.join(ignore_list) }]\nfocus_prompts: [{ ', '.join(focus_list) }]\nthreshold: {MASK_THRESHOLD}\ndilate: 5\n"
            with open("configs/sam_mask.yaml", "w", encoding="utf-8") as f: f.write(mask_yaml_content)

            # NOTE: `prompts`/`focus_prompts` in configs/sam_mask.yaml are still honored as legacy keys.
            print("🚀 Running `make mask`...")
            run_step("make mask", ["uv", "run", "--no-sync", "make", "mask"])

        print("🚀  Running preprocess-vae...")
        run_step("preprocess-vae", ["uv", "run", "--no-sync", "make", "preprocess-vae"], T4_HINT)

        if USE_SIGMA_LOWRES:
            print("🚀  Running preprocess-demote for sigma_lowres (1024:896, 1024:768)...")
            run_step("preprocess-demote", ["uv", "run", "--no-sync", "make", "preprocess-demote", "ARGS=--sigma_demote 1024:896,1024:768"], T4_HINT)

        print("🚀  Running preprocess-te...")
        # NOTE: tasks.py already passes a default --dit (anima-base); the explicit
        # --dit here overrides it (argparse last-wins) so DIFFUSION_MODEL_NAME is honored.
        run_step(
            "preprocess-te",
            ["uv", "run", "--no-sync", "make", "preprocess-te", f"ARGS=--dit models/diffusion_models/{DIFFUSION_MODEL_NAME}"],
            f"{T4_HINT} {FLASH_HINT}",
        )

        if USE_CMMD:
            print("\n🚀 Caching Perceptual Features for CMMD Validation...")
            run_step("preprocess-pe (CMMD)", ["uv", "run", "--no-sync", "make", "preprocess-pe"], T4_HINT)

        if USE_REPA:
            print("\n🚀 Caching REPA Spatial PE features...")
            run_step("preprocess-pe (REPA)", ["uv", "run", "--no-sync", "make", "preprocess-pe", "ARGS=--encoder pe_spatial"], T4_HINT)
    # Gate: never mark complete with missing TE caches (e.g. a crashed preprocess-te
    # must not auto-skip next run, otherwise training would start on incomplete caches).
    _resized_imgs = [p for p in glob.glob("post_image_dataset/resized/**/*", recursive=True) if os.path.isfile(p) and p.lower().endswith(IMAGE_EXTS)]
    _te_caches = [p for p in glob.glob("post_image_dataset/lora/**/*_anima_te.safetensors", recursive=True) if os.path.isfile(p)]
    if _resized_imgs and len(_te_caches) < len(_resized_imgs):
        raise SystemExit(f"💥 TE cache incomplete ({len(_te_caches)}/{len(_resized_imgs)}). Marker NOT written — re-run Cell ① with FORCE_REPREPROCESS=True.")
    # Marker to auto-skip preprocessing on next run
    with open(MARKER_FILE, "w", encoding="utf-8") as f:
        f.write("completed")
    print(f"\n📌 Preprocessing will auto-skip next run (reset via FORCE_REPREPROCESS)")
    print("\n✅ Preprocess done. Optional: run the SAM3 mask check, then Cell ② (Training).")

run_preprocess()

In [ ]:
# @title ✔️ SAM3 Mask Test (Optional, 可跳过)
# @markdown 仅当 Cell ① 开启 `USE_MASKED_LOSS` 并已生成 mask 时需要运行。默认折叠，可直接跳过进 Cell ②。
RUN_MASK_TEST = True # @param {type:"boolean"}
# Max images to show (0 = no limit)
MAX_DISPLAY_COUNT = 10 # @param {type:"integer"}
import os
import glob

# Dirs to inspect
RESIZED_DIR = "post_image_dataset/resized"
MASK_DIR = "post_image_dataset/masks"

# Carry flag from Cell ① (fallback for standalone runs)
if "USE_MASKED_LOSS" not in globals():
    USE_MASKED_LOSS = False

if not RUN_MASK_TEST:
    print("⏭️ [Skip] RUN_MASK_TEST=False, 直接进 Cell ② 训练即可。")
elif not USE_MASKED_LOSS:
    print("⏭️ [Skip] Cell ① USE_MASKED_LOSS=False，未生成 mask，直接进 Cell ② 训练即可。")
# Find all images incl. subfolders
else:
    import matplotlib.pyplot as plt
    from PIL import Image
    import numpy as np

    image_paths = glob.glob(os.path.join(RESIZED_DIR, "**", "*.*"), recursive=True)
    image_paths = [p for p in image_paths if p.lower().endswith(('.png', '.jpg', '.jpeg', '.webp'))]
    image_paths.sort()

    if not image_paths:
        print(f"❌ {RESIZED_DIR} No images in {RESIZED_DIR} or subfolders.")
    elif not os.path.exists(MASK_DIR) or not os.listdir(MASK_DIR):
        print(f"❌ {MASK_DIR} No masks in {MASK_DIR}. Generate masks first.")
    else:
        display_count = 0

        for img_p in image_paths:
            if MAX_DISPLAY_COUNT > 0 and display_count >= MAX_DISPLAY_COUNT:
                print(f"\n⚠️ Showing max {MAX_DISPLAY_COUNT} images. Increase MAX_DISPLAY_COUNT for more.")
                break

            rel_path = os.path.relpath(img_p, RESIZED_DIR)
            rel_stem, _ = os.path.splitext(rel_path)

            # Keep subfolder path for mask filename
            mask_p = os.path.join(MASK_DIR, f"{rel_stem}_mask.png")
            base_name = os.path.basename(img_p)

            if not os.path.exists(mask_p):
                folder_hint = os.path.dirname(rel_path)
                print(f"⚠️ [{folder_hint}] {base_name}mask missing, skipped.")
                continue

            # Load image and mask
            img = Image.open(img_p).convert("RGB")
            mask = Image.open(mask_p).convert("L")

            # Red overlay layer
            mask_np = np.array(mask)
            overlay_np = np.zeros((mask_np.shape[0], mask_np.shape[1], 4), dtype=np.uint8)

            # Ignored area (0) = translucent red
            overlay_np[mask_np == 0] = [255, 0, 0, 120]
            overlay_img = Image.fromarray(overlay_np, "RGBA")

            # Overlay red filter on original
            img_rgba = img.convert("RGBA")
            visualized_img = Image.alpha_composite(img_rgba, overlay_img)

            # ----------------------------------------------------
            # Display
            # ----------------------------------------------------
            fig, axes = plt.subplots(1, 3, figsize=(15, 5))

            axes[0].imshow(img)
            # Show subfolder in title
            axes[0].set_title(f"Original ({os.path.dirname(rel_path)} / {base_name})", fontsize=10)
            axes[0].axis("off")

            axes[1].imshow(mask, cmap="gray")
            axes[1].set_title("SAM3 Binary Mask\n(White=Train, Black=Ignore)", fontsize=11)
            axes[1].axis("off")

            axes[2].imshow(visualized_img)
            axes[2].set_title("Mask Overlay Result\n(Red Area = Loss Ignored)", fontsize=11, color="red")
            axes[2].axis("off")

            plt.tight_layout()
            plt.show()

            display_count += 1

In [ ]:
# @title ② Training (tune params + re-run freely)

# @markdown ### ▶️ Resume (Optional): Path to an existing LoRA (`.safetensors`) to resume from. Leave empty for fresh training.
NETWORK_WEIGHTS = "" # @param {type:"string"}

# @markdown ### ▶️ Epoches
MAX_TRAIN_EPOCHS = 10 # @param {type:"integer"}
SAVE_EVERY_N_EPOCHS = 1 # @param {type:"integer"}

# @markdown ### ▶️ Learning
TRAIN_BATCH_SIZE = 1 # @param {type:"integer"}
GRADIENT_ACCUMULATION_STEPS = 2 # @param {type:"integer"}
LEARNING_RATE = "5e-5" # @param {type:"string"}
OPTIMIZER_TYPE = "AdamW" # @param ["AdamW", "Prodigy", "ProdigyPlusScheduleFree", "CAME"]
LR_SCHEDULER = "cosine" # @param ["constant", "cosine", "linear", "rex"]
LR_WARMUP_RATIO = 0.05 # @param {type:"slider", min:0.0, max:0.2, step:0.01}
TIMESTEP_SAMPLING = "sigmoid" #@param ["uniform", "sigmoid", "shift", "flux_shift"]
CAPTION_DROPOUT_RATE = 0.1 # @param {type:"number"}

# @markdown ### ▶️ Structure (LoRA)
NETWORK_DIM = 32 # @param {type:"integer"}
NETWORK_ALPHA = 32 # @param {type:"integer"}

# @markdown ### ▶️ Validation
# @markdown Validation image count (0 = skip validation)
VALIDATION_SPLIT_NUM = 0 # @param {type:"integer"}

# Hidden training defaults (upstream README default-true path + compat)
CHECKPOINTING_EPOCHS = 9999
BLOCKS_TO_SWAP = 0
MAX_GRAD_NORM = 8.0
TORCH_COMPILE = True
COMPILE_INDUCTOR_MODE = "default"
ATTN_MODE = "torch"
GRADIENT_CHECKPOINTING = False
# Recomputes activations in backward instead of storing them (big VRAM cut).
# The unsloth variant additionally streams checkpointed tensors to CPU RAM —
# maximum savings, but the GPU then idles on PCIe transfers (that was the
# '6GB + low util' mode). Keep False while VRAM allows; flip True only on OOM.
UNSLOTH_OFFLOAD_CHECKPOINTING = False
WEIGHT_SVD = True
TRAIN_ADALN = True
ADALN_RANK = 8
ADALN_ALPHA = 0
USE_TIMESTEP_MASK = False
LORAPLUS_LR_RATIO = 0
SIGMOID_SCALE = 1.0
DISCRETE_FLOW_SHIFT = 1.0
prodigy_d_coef = 1.0
prodigy_d0 = 1e-6
prodigy_schedulefree_c = 0
LR_SCHEDULER_NUM_CYCLES = 4
RESTARTS_GAMMA = 0.8
RESTARTS_CYCLE_MULTIPLIER = 1.0
RESTARTS_FIRST_CYCLE_STEPS = 0
RESTARTS_MIN_LR = 1e-6
RESTARTS_D = 0.9
VALIDATE_EVERY_N_EPOCHS = 2
DETERMINISTIC = False
VALIDATION_SAMPLE_STEPS = 20
VALIDATION_CFG_SCALE = 1.0
USE_SOUP = False
UNCOND_INIT_ONLY = False
SOUP_PATH_PATTERN = "1_target/*"
SOUP_UNCOND_EPOCHS = 4
SOUP_UNCOND_RATIO = 1.0
SOUP_UNCOND_LR = "2e-5"
SOUP_UNCOND_DIM = 32
SOUP_UNCOND_ALPHA = 128

import os
import shlex
import shutil
import subprocess
import re
import glob
import math

# Carry settings from Cells ⓪① (fallbacks allow standalone runs)
if "PROJECT_NAME" not in globals(): PROJECT_NAME = ""
if "USE_VOCAB_PACK" not in globals(): USE_VOCAB_PACK = False
if "DIFFUSION_MODEL_NAME" not in globals(): DIFFUSION_MODEL_NAME = "anima-base-v1.0.safetensors"
if "IS_T4_GPU" not in globals(): IS_T4_GPU = True
if "MIXED_PRECISION" not in globals(): MIXED_PRECISION = "fp16"
if "WANDB_API_KEY" not in globals(): WANDB_API_KEY = ""
if "HF_TOKEN" not in globals(): HF_TOKEN = ""
if "USE_MASKED_LOSS" not in globals(): USE_MASKED_LOSS = False
if "USE_SIGMA_LOWRES" not in globals(): USE_SIGMA_LOWRES = True
if "USE_REPA" not in globals(): USE_REPA = False
if "USE_CMMD" not in globals(): USE_CMMD = False
if "NUM_REPEATS" not in globals(): NUM_REPEATS = 10

# --- T4 (16GB) memory envelope -------------------------------------------------
# Applied after every other carry-in; nothing later overrides these two:
#   - GRADIENT_CHECKPOINTING: without checkpointing, activations of all 28 blocks
#     at 1024x1024 do not fit 16GB (train.py comment: "a 4k-token backward without
#     it does not fit the 16 GB envelope"); the stock `low_vram` preset is it.
#   - TORCH_COMPILE: measured SLOWER on a T4 even with checkpointing (dynamic
#     seq families recompile per shape; "Not enough SMs" disables max-autotune,
#     so inductor emits generic kernels for the 40-SM Turing card). The fp32
#     block stack is mandatory on T4 anyway (fp16 residual overflows -> zero-weight
#     LoRA, see restore_fp32_block_path()), which makes eager + SDPA the fastest
#     configuration available. base.toml torch_compile is patched to false below.
if IS_T4_GPU:
    GRADIENT_CHECKPOINTING = True
    TORCH_COMPILE = False
    print("⚙️ [T4] memory envelope: GRADIENT_CHECKPOINTING=True, TORCH_COMPILE=False")

assert PROJECT_NAME.strip(), "Set PROJECT_NAME in Cell ① and run it first."
assert os.path.exists("/content/anima_lora"), "Run Cell ⓪ (Environment Setup) first."
os.chdir("/content/anima_lora")
DRIVE_BASE = "/content/drive/MyDrive/lora_training"
DATASET_FOLDER = os.path.join(DRIVE_BASE, "datasets", PROJECT_NAME)
DATASET_ZIP = DATASET_FOLDER + ".zip"
OUTPUT_DIR = os.path.join(DRIVE_BASE, "output", PROJECT_NAME)
LOCAL_DATASET = "/content/anima_lora/image_dataset"
MARKER_FILE = os.path.join("/content/anima_lora", f".cache_done_{PROJECT_NAME}")
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.environ['PYTHONWARNINGS'] = "ignore"
if HF_TOKEN.strip():
    os.environ['HF_TOKEN'] = HF_TOKEN

# Dataset size (reuse Cell ① effective count when available; N_ folder overrides NUM_REPEATS)
NUM_REPEATS = int(globals().get("NUM_REPEATS", 10))
if "count_effective_images" in globals():
    num_images_raw = count_dataset_images(LOCAL_DATASET)
    if num_images_raw == 0:
        num_images_raw = count_dataset_images(DATASET_FOLDER)
    num_images = count_effective_images(LOCAL_DATASET, NUM_REPEATS)
    if num_images == 0:
        num_images = count_effective_images(DATASET_FOLDER, NUM_REPEATS)
else:
    import re as _re_steps
    _exts = ('.png', '.jpg', '.jpeg', '.webp', '.bmp')
    def _rep_of_rel_steps(rel_dir, default):
        if not rel_dir or rel_dir == "." or rel_dir.startswith(".."):
            return default
        for part in reversed(_re_steps.split(r"[\\/]", rel_dir)):
            m = _re_steps.match(r"^(\d+)_.*$", part)
            if m:
                return int(m.group(1))
        return default
    num_images_raw = 0
    num_images = 0
    for _base in (LOCAL_DATASET, DATASET_FOLDER):
        if num_images_raw > 0:
            break
        if not os.path.exists(_base):
            continue
        for _root, _, _fs in os.walk(_base):
            _rel = os.path.relpath(_root, _base)
            _rep = _rep_of_rel_steps(_rel, NUM_REPEATS)
            for _f in _fs:
                if _f.lower().endswith(_exts):
                    num_images_raw += 1
                    if _rep != 0:
                        num_images += _rep
NUM_IMAGES = num_images

effective_batch_size = TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS

if num_images > 0:
    steps_per_epoch = math.ceil(num_images / effective_batch_size)
    total_steps = steps_per_epoch * MAX_TRAIN_EPOCHS
    calculated_warmup_steps = int(total_steps * LR_WARMUP_RATIO)
    print(f"📊 [Dataset analysis] (effective = N_ folder if present else NUM_REPEATS={NUM_REPEATS})")
    print(f" ├─ Raw images: {num_images_raw}")
    print(f" ├─ Effective samples/epoch: {num_images}")
    print(f" ├─ Effective batch (Batch x Accum): {effective_batch_size}")
    print(f" ├─ Steps per epoch: {steps_per_epoch} steps")
    print(f" ├─ Total steps ({MAX_TRAIN_EPOCHS} epochs): {total_steps} steps")
    print(f" └─ Warmup steps ({LR_WARMUP_RATIO*100:.0f}%): {calculated_warmup_steps} steps")
    if VALIDATION_SPLIT_NUM > 0:
        print(f"    ⚠️ validation_split_num={VALIDATION_SPLIT_NUM}: val images are carved out before repeat expansion, actual train steps will be slightly lower (estimate above is an upper bound).")
else:
    calculated_warmup_steps = 100
# Build train_args

# Optimizer
optimizer_args = []
max_grad_norm_cli = f"--max_grad_norm {MAX_GRAD_NORM} "

if OPTIMIZER_TYPE == "AdamW":
    optimizer_args = ["weight_decay=0.1", "betas=0.9,0.99", "fused=True"]
elif OPTIMIZER_TYPE == "ProdigyPlusScheduleFree":
    OPTIMIZER_TYPE = "prodigyplus.ProdigyPlusScheduleFree"
    optimizer_args = ["betas=0.95,0.99", f"d_coef={prodigy_d_coef}", f"d0={prodigy_d0}"]
    if prodigy_schedulefree_c > 0:
        optimizer_args.append(f"schedulefree_c={prodigy_schedulefree_c}")
    max_grad_norm_cli = "--max_grad_norm 0.0 "
elif OPTIMIZER_TYPE == "Prodigy":
    optimizer_args = [f"d_coef={prodigy_d_coef}", f"d0={prodigy_d0}", "betas=0.9,0.99", "weight_decay=0.01", "decouple=True", "use_bias_correction=True", "safeguard_warmup=True"]
elif OPTIMIZER_TYPE == "Prodigy_adv":
    OPTIMIZER_TYPE = "adv_optm.optim.Prodigy_adv.Prodigy_adv"
    optimizer_args = [
        "betas=0.9,0.99", "nesterov=False",
        f"d_coef={prodigy_d_coef}", f"d0={prodigy_d0}", "d_limiter=False",
        "weight_decay=0.01", "cautious_wd=True",
        "orthogonal_gradient='disabled'",
        "spectral_normalization=False",
        "kourkoutas_beta=False",
        "state_precision='fp32'",
        "compiled_optimizer=True",
    ]
elif OPTIMIZER_TYPE == "SinkSGD_adv":
    OPTIMIZER_TYPE = "adv_optm.optim.SinkSGD_adv.SinkSGD_adv"
    optimizer_args = [
        "momentum=0.95", "nesterov=True", "normed_momentum=True", "snr_cond=True",
        "weight_decay=0.1", "geometric_wd=False", "cautious_wd=True",
        "orthogonal_sinkhorn=False",
        "orthogonal_gradient='disabled'",
        "spectral_normalization=False",
        "state_precision='fp32'",
        "compiled_optimizer=False",
    ]
elif OPTIMIZER_TYPE == "AdamW_adv":
    OPTIMIZER_TYPE = "adv_optm.optim.AdamW_adv.AdamW_adv"
    optimizer_args = [
        "betas=0.9,0.99", "nesterov=False", "normed_momentum=True",
        "weight_decay=0.01", "cautious_wd=True",
        "orthogonal_gradient='disabled'",
        "spectral_normalization=False",
        "use_atan2=False",
        "kourkoutas_beta=True", "ema_alpha=0.95",
        "state_precision='fp32'",
        "compiled_optimizer=False",
    ]
elif "AdamWScheduleFreePlus" in OPTIMIZER_TYPE:
    OPTIMIZER_TYPE = "LoraEasyCustomOptimizer.adamw_schedulefree_plus.AdamWScheduleFreePlus"
    optimizer_args = [
        "weight_decay=5",
        "polyak_f_ema=0.95",
        "max_polyak_lr=5",
        "polyak_beta=0.95",
        "r=2.0",
        f"warmup_steps={calculated_warmup_steps}",
        f"c_warmup={calculated_warmup_steps * 2}",
        f"sf_beta1_anneal_steps={calculated_warmup_steps * 3}"
    ]
    max_grad_norm_cli = "--max_grad_norm 0.0 "
elif OPTIMIZER_TYPE == "CAME":
    OPTIMIZER_TYPE = "pytorch_optimizer.optimizer.came.CAME"
    optimizer_args = ["weight_decay=0.01"]
elif OPTIMIZER_TYPE == "WarpAINO":
    OPTIMIZER_TYPE = "LoraEasyCustomOptimizer.warpaino.WarpAINO"
    optimizer_args = [
        "weight_decay=0.1", "relative_wd=True", "cautious_wd=True",
        "meta_lr=0",
        "came_confidence=True",
        "nesterov_value=False",
        "cautious_update=True",
        "compile_step=False", "foreach=True",
        "ortho_dtype=torch.float32", "warp_dtype=torch.float32"
    ]
elif OPTIMIZER_TYPE == "WarpAINO_legacy":
    OPTIMIZER_TYPE = "LoraEasyCustomOptimizer.warpaino_legacy.WarpAINO_legacy"
    optimizer_args = [
        "weight_decay=0.01", "relative_wd=True",
        "meta_lr=0.05",
        "compile_step=False", "foreach=True",
        "ortho_dtype=torch.float32", "warp_dtype=torch.float32"
    ]
elif OPTIMIZER_TYPE == "WarpAdam":
    OPTIMIZER_TYPE = "LoraEasyCustomOptimizer.warpadam.WarpAdam"
    optimizer_args = ["meta_lr=0.001"]

elif OPTIMIZER_TYPE == "RelAdamW":
    OPTIMIZER_TYPE = "LoraEasyCustomOptimizer.RelAdamW.RelAdamW"
    optimizer_args = ["weight_decay=0.1"]

optimizer_args_cli = "--optimizer_args " + " ".join([f'"{arg}"' for arg in optimizer_args]) + " "

# Network Args
network_args_list = []
network_args_list.extend([
    f"use_timestep_mask={USE_TIMESTEP_MASK}", "min_rank=1", "alpha_rank_scale=1.0",
    f"use_repa={USE_REPA}",
])

network_args_list.append("down_init=weight_svd" if WEIGHT_SVD else "down_init=kaiming")

if LORAPLUS_LR_RATIO >= 1:
    network_args_list.append(f"loraplus_lr_ratio={LORAPLUS_LR_RATIO}")

print(f"✅ LoRA options applied: use_timestep_mask={USE_TIMESTEP_MASK}, use_repa={USE_REPA}, weight_svd={WEIGHT_SVD}, train_adaln={TRAIN_ADALN}")

network_args_cli = ""
if network_args_list:
    args_joined = " ".join([f"'{arg}'" for arg in network_args_list])
    network_args_cli = f"--network_args {args_joined} "

# WandB
wandb_cli = ""
if WANDB_API_KEY.strip():
    wandb_cli = (
        f'--log_with wandb '
        f'--log_tracker_name "Anima_LoRA_Project" '
        f'--wandb_run_name "{PROJECT_NAME}_Run" '
        f'--wandb_api_key "{WANDB_API_KEY.strip()}" '
        f'--log_every_n_steps 1 '
    )

# Validation
validation_cli = (
    f'--validation_split_num {VALIDATION_SPLIT_NUM} '
    f'--validate_every_n_epochs {VALIDATE_EVERY_N_EPOCHS} '
    f'--validation_sample_steps {VALIDATION_SAMPLE_STEPS} '
    f'--validation_cfg_scale {VALIDATION_CFG_SCALE} '
)
if USE_CMMD:
    validation_cli += '--use_cmmd '
else:
    validation_cli += '--no-use_cmmd '

deterministic_cli = ""
if DETERMINISTIC:
    deterministic_cli = "--deterministic --paired_step_rng --seed 1557 "

# Custom scheduler (CosineAnnealingWarmRestarts / RexAnnealingWarmRestarts) args
lr_scheduler_type_cli = ""
lr_scheduler_args_cli = ""

if LR_SCHEDULER in ["CosineAnnealingWarmRestarts", "RexAnnealingWarmRestarts", "rex"]:
    if LR_SCHEDULER == "CosineAnnealingWarmRestarts":
        lr_scheduler_type = "LoraEasyCustomOptimizer.CosineAnnealingWarmRestarts.CosineAnnealingWarmRestarts"
        scheduler_args = [
            f"gamma={RESTARTS_GAMMA}",
            f"cycle_multiplier={RESTARTS_CYCLE_MULTIPLIER}",
            f"first_cycle_max_steps={RESTARTS_FIRST_CYCLE_STEPS}",
            f"min_lr={RESTARTS_MIN_LR}"
        ]
        lr_scheduler_type_cli = f'--lr_scheduler_type "{lr_scheduler_type}" '
    elif LR_SCHEDULER == "RexAnnealingWarmRestarts":
        lr_scheduler_type = "LoraEasyCustomOptimizer.RexAnnealingWarmRestarts.RexAnnealingWarmRestarts"
        scheduler_args = [
            f"gamma={RESTARTS_GAMMA}",
            f"cycle_multiplier={RESTARTS_CYCLE_MULTIPLIER}",
            f"first_cycle_max_steps={RESTARTS_FIRST_CYCLE_STEPS}",
            f"min_lr={RESTARTS_MIN_LR}",
            f"d={RESTARTS_D}"
        ]
        lr_scheduler_type_cli = f'--lr_scheduler_type "{lr_scheduler_type}" '
    elif LR_SCHEDULER == "rex":
        scheduler_args = [f"d={RESTARTS_D}"]

    lr_scheduler_args_cli = "--lr_scheduler_args " + " ".join([f'"{arg}"' for arg in scheduler_args]) + " "


# sigma_lowres
# NOTE: there is deliberately no `--sigma_lowres*` CLI block here. It used to be
# built into a `sigma_lowres_cli` string that was NEVER spliced into train_args,
# so sigma-demotion only ever turned on through the `sigma_lowres` key in
# configs/base.toml (patched below), which carries the same route/threshold/span
# values. Left as a comment so nobody re-adds the dead variable -- note that
# `--sigma_lowres_yarnsig` takes no value, so wiring it in would have eaten the
# following argv token.

# etc.
compile_args_cli = f"--torch_compile --compile_inductor_mode {COMPILE_INDUCTOR_MODE} " if TORCH_COMPILE else ""
grad_ckpt_cli = "--gradient_checkpointing " if GRADIENT_CHECKPOINTING else ""
unsloth_offload_checkpointing_cli = "--unsloth_offload_checkpointing " if GRADIENT_CHECKPOINTING and UNSLOTH_OFFLOAD_CHECKPOINTING else ""
save_precision_cli = '--save_precision "fp16" ' if IS_T4_GPU else '--save_precision "bf16" '
network_weights_cli = f'--network_weights "{NETWORK_WEIGHTS.strip()}" ' if NETWORK_WEIGHTS.strip() else ""

train_args = (
    f'--pretrained_model_name_or_path "models/diffusion_models/{DIFFUSION_MODEL_NAME}" '
    f'--output_dir "{OUTPUT_DIR}" '
    f'--output_name "{PROJECT_NAME}" '
    f'--train_batch_size {TRAIN_BATCH_SIZE} '
    f'--network_dim {NETWORK_DIM} '
    f'--network_alpha {NETWORK_ALPHA} '
    f'--learning_rate {LEARNING_RATE} '
    f'--max_train_epochs {MAX_TRAIN_EPOCHS} '
    f'--save_every_n_epochs {SAVE_EVERY_N_EPOCHS} '
    f'--checkpointing_epochs {CHECKPOINTING_EPOCHS} '
    f'--gradient_accumulation_steps {GRADIENT_ACCUMULATION_STEPS} '
    f'{grad_ckpt_cli}'
    f'{unsloth_offload_checkpointing_cli}'
    f'--optimizer_type "{OPTIMIZER_TYPE}" '
    f'{optimizer_args_cli}'
    f'{max_grad_norm_cli}'
    f'{network_args_cli}'
    f'{compile_args_cli}'
    f'--lr_scheduler "{LR_SCHEDULER}" '
    f'--lr_scheduler_num_cycles {LR_SCHEDULER_NUM_CYCLES} '
    f'--lr_warmup_steps {LR_WARMUP_RATIO} '
    f'{lr_scheduler_type_cli}'
    f'{lr_scheduler_args_cli}'
    f'--timestep_sampling {TIMESTEP_SAMPLING} '
    f'--sigmoid_scale {SIGMOID_SCALE} '
    f'--discrete_flow_shift {DISCRETE_FLOW_SHIFT} '
    f'--blocks_to_swap {BLOCKS_TO_SWAP} '
    f'--mixed_precision "{MIXED_PRECISION}" '
    f'{save_precision_cli}'
    f'--attn_mode "{ATTN_MODE}" '
    f'{network_weights_cli}'
    f'--caption_dropout_rate {CAPTION_DROPOUT_RATE} '
    f'--console_log_simple '
    f'{validation_cli}'
    f'{deterministic_cli}'
    f'{wandb_cli}'
)

# ==============================================================================
# Patches and orchestration
# ==============================================================================
def apply_general_optimizations():

    # base.toml
    base_toml_path = "configs/base.toml"
    if os.path.exists(base_toml_path):
        with open(base_toml_path, "r", encoding="utf-8") as f: toml_content = f.read()

        target_model_path = f"models/diffusion_models/{DIFFUSION_MODEL_NAME}"
        new_toml_content = re.sub(
            r'pretrained_model_name_or_path\s*=\s*".*"',
            f'pretrained_model_name_or_path = "{target_model_path}"',
            toml_content
        )

        new_toml_content = re.sub(r'batch_size\s*=\s*\d+', f'batch_size = {TRAIN_BATCH_SIZE}', new_toml_content)
        new_toml_content = re.sub(r'gradient_accumulation_steps\s*=\s*\d+', f'gradient_accumulation_steps = {GRADIENT_ACCUMULATION_STEPS}', new_toml_content)
        new_toml_content = re.sub(r'validation_split_num\s*=\s*\d+', f'validation_split_num = {VALIDATION_SPLIT_NUM}', new_toml_content)
        # Dataset repeats: global NUM_REPEATS from Cell ①; N_ folders override it at train time (repeat_by_folder_name).
        _num_repeats = int(globals().get("NUM_REPEATS", 10))
        new_toml_content = re.sub(r'repeat_by_folder_name\s*=\s*(true|false)', 'repeat_by_folder_name = true', new_toml_content)
        new_toml_content = re.sub(r'num_repeats\s*=\s*\d+', f'num_repeats = {_num_repeats}', new_toml_content, count=1)

        adaln_bool = "true" if TRAIN_ADALN else "false"
        target_rank = ADALN_RANK if TRAIN_ADALN else 0
        target_alpha = ADALN_ALPHA if TRAIN_ADALN else 0

        new_toml_content = re.sub(r'train_adaln\s*=\s*(true|false)', f'train_adaln = {adaln_bool}', new_toml_content)
        new_toml_content = re.sub(r'adaln_rank\s*=\s*\d+(?:\n\s*adaln_alpha\s*=\s*\d+)?', f'adaln_rank = {target_rank}\nadaln_alpha = {target_alpha}', new_toml_content)

        sigma_lowres_bool = "true" if USE_SIGMA_LOWRES else "false"
        new_toml_content = re.sub(r'sigma_lowres\s*=\s*(true|false)', f'sigma_lowres = {sigma_lowres_bool}', new_toml_content)

        # `mixed_precision` / `save_precision` are deliberately NOT patched here:
        # the fork dropped both keys from configs/base.toml, so these regexes were
        # silently no-ops. They still take effect via `--mixed_precision` /
        # `--save_precision` in train_args (CLI beats base.toml in the merge chain).

        new_toml_content = re.sub(r'attn_mode\s*=\s*".*"', f'attn_mode = "{ATTN_MODE}"', new_toml_content)

        if not TORCH_COMPILE:
            new_toml_content = re.sub(r'torch_compile\s*=\s*true', 'torch_compile = false', new_toml_content)
        if USE_MASKED_LOSS:
            new_toml_content = re.sub(r'masked_loss\s*=\s*(true|false)', 'masked_loss = true', new_toml_content)
            if "mask_dir" in new_toml_content:
                new_toml_content = re.sub(r'mask_dir\s*=\s*".*"', 'mask_dir = "post_image_dataset/masks"', new_toml_content)
            else:
                new_toml_content = new_toml_content.replace('[[dataset.subsets]]', '[[dataset.subsets]]\n  mask_dir = "post_image_dataset/masks"')
        if toml_content != new_toml_content:
            with open(base_toml_path, "w", encoding="utf-8") as f: f.write(new_toml_content)
            print("  ✅ [OK] base.toml patched")
        else:
            print("  ⏭️ [Skip] base.toml already patched")

    if USE_SOUP:
        # Patch params in configs/soup/soup.toml
        soup_toml_path = "configs/soup/soup.toml"
        if os.path.exists(soup_toml_path):
            with open(soup_toml_path, "r", encoding="utf-8") as f: content = f.read()
            new_content = re.sub(r'use_repa\s*=\s*\w+', f'use_repa = {str(USE_REPA).lower()}', content)
            new_content = re.sub(r'learning_rate\s*=\s*[\d\.e\-]+', f'learning_rate = {SOUP_UNCOND_LR}', new_content)
            new_content = re.sub(r'network_dim\s*=\s*\d+', f'network_dim = {SOUP_UNCOND_DIM}', new_content)
            new_content = re.sub(r'network_alpha\s*=\s*[\d\.]+', f'network_alpha = {SOUP_UNCOND_ALPHA}', new_content)

            with open(soup_toml_path, "w", encoding="utf-8") as f: f.write(new_content)
            print(f"  ✅ [OK] soup.toml patched (use_repa={USE_REPA}, LR:{SOUP_UNCOND_LR}, DIM:{SOUP_UNCOND_DIM}, ALPHA:{SOUP_UNCOND_ALPHA})")


def apply_t4_and_fp16_patches():
    # Mirrors Cell ① ensure_t4_compat: T4 (Turing) cannot run FlashAttention.
    # Uninstall from .venv (never `uv remove`, which edits pyproject.toml) and VERIFY.
    # Self-contained (no dependency on Cell ①) so Cell ② runs standalone.
    print("\n🚀 Removing flash-attn...")
    try:
        _p = subprocess.run(
            [".venv/bin/python", "-c", "import importlib.util; print(importlib.util.find_spec('flash_attn'))"],
            capture_output=True, text=True, timeout=120,
        )
        _spec = (_p.stdout or "").strip() if _p.returncode == 0 else "UNKNOWN"
    except Exception as e:
        _spec = f"UNKNOWN({e})"
    if _spec == "None":
        print("  ✅ flash-attn already absent (SDPA fallback active)")
        return
    print(f"   found: {_spec}; uninstalling...")
    _un = subprocess.run(
        ["uv", "pip", "uninstall", "--python", ".venv/bin/python", "-y", "flash-attn"],
        capture_output=True, text=True, timeout=300,
    )
    print(((_un.stdout or "") + "\n" + (_un.stderr or "")).strip()[-2000:])
    if _un.returncode != 0:
        raise SystemExit(f"💥 [T4] failed to uninstall flash-attn (exit {_un.returncode}). Fix manually, then re-run Cell ②.")
    _p2 = subprocess.run(
        [".venv/bin/python", "-c", "import importlib.util; print(importlib.util.find_spec('flash_attn'))"],
        capture_output=True, text=True, timeout=120,
    )
    if (_p2.stdout or "").strip() != "None":
        raise SystemExit(f"💥 [T4] flash-attn still importable after uninstall: {(_p2.stdout or '').strip()}. Uninstall manually and re-run Cell ②.")
    print("  ✅ flash-attn removed and verified")

print("\n\U0001f6e0️ Applying internal patches")
apply_general_optimizations()
# Re-pin vocab_pack AFTER apply_general_optimizations() rewrites base.toml.
set_vocab_pack(USE_VOCAB_PACK)
# Restore the fp16 block-stack escape hatch. Without it the run completes
# and saves a ZERO-weight LoRA on a T4 -- see restore_fp32_block_path().
restore_fp32_block_path()
if IS_T4_GPU:
    apply_t4_and_fp16_patches()

def run_training():
    if USE_SOUP:
        print("\n=== [3/3] Starting LoRA Soup Training Pipeline ===")
        print(f"🥣 Pipeline: uncond pre-train (Pool) -> multi-seed fine-tune -> ΔW SVD merge")

        soup_train_args = train_args

        if not UNCOND_INIT_ONLY:
            # Strip output path/name args to protect Soup temp-file layout.
            soup_train_args = re.sub(r'--output_dir\s+[^\s]+', '', soup_train_args)
            soup_train_args = re.sub(r'--output_name\s+[^\s]+', '', soup_train_args)
            # Strip seed arg for Soup multi-seed fine-tune.
            soup_train_args = re.sub(r'--seed\s+[^\s]+', '', soup_train_args)

        !uv run --no-sync python scripts/soup/pipeline.py --preset default --path_pattern "{SOUP_PATH_PATTERN}" --uncond_epochs {SOUP_UNCOND_EPOCHS} --uncond_ratio {SOUP_UNCOND_RATIO} {soup_train_args}

        # Resolve paths dynamically from project vars.
        print(f"\n🚚 Copying final master Soup model to Google Drive...")

        # Create project folder if missing.
        !mkdir -p {OUTPUT_DIR}

        # Find all anima_soup files.
        soup_files = glob.glob("/content/anima_lora/output/ckpt/anima_soup_*.safetensors")

        master_files = [f for f in soup_files if not re.search(r'_s\d+\.safetensors', f)]

        if master_files:
            master_soup_file = master_files[0]
            target_path = f"{OUTPUT_DIR}/{PROJECT_NAME}_soup.safetensors"
            shutil.copy2(master_soup_file, target_path)
            print(f"✅ [OK] Soup model saved: {target_path}")
        else:
            print("❌ [Error] Merged Soup file not found.")

    else:
        print("\n=== [3/3] Starting Standard Plain LoRA Model Training ===")
        # run_step, not the IPython `!` magic: the magic swallows the exit code, so a
        # crashed train.py used to fall through silently and leave a stale/zero LoRA.
        run_step(
            "train",
            ["uv", "run", "--no-sync", "python", "train.py", "--method", "lora",
             "--preset", "default"] + shlex.split(train_args),
            "See output/notebook_logs/train.log. If the tail shows torch.OutOfMemoryError, "
            "next levers (in order): lower the bucket resolution in Cell ① "
            "(e.g. TARGET_RES='512 768 896') with FORCE_REPREPROCESS=True, then re-run "
            "Cell ②; or set UNSLOTH_OFFLOAD_CHECKPOINTING=True (requires blocks_to_swap "
            "=0, which is already the case) if it is not on yet. If the loss is nan/inf instead, the fp16 "
            "block stack is still unprotected -- see restore_fp32_block_path().",
        )

run_training()

if not USE_SOUP:
    # `lora_up` is zero-initialised, so a run whose every optimizer step got skipped
    # by accelerate's GradScaler still saves a plausible-looking file full of exact
    # zeros. That is invisible at inference ("the LoRA does nothing") and is exactly
    # what happened here -- make it a hard stop instead.
    assert_lora_actually_trained(OUTPUT_DIR)
